# Get Data

## ClinicalTrials.gov API Resources

Here are some useful resources for understanding and using the ClinicalTrials.gov API:

*   **[ClinicalTrials.gov Data API Reference](https://clinicaltrials.gov/data-api/api#extapi)**:
    This is the main entry point for the API documentation. It provides details on endpoints, request parameters, response formats, and authentication (if applicable).

*   **[ClinicalTrials.gov Data API - Search Areas](https://clinicaltrials.gov/data-api/about-api/search-areas)**:
    This page describes the various fields and parameters you can use to refine your searches. Understanding these "search areas" is crucial for constructing effective queries to find specific clinical trials.

*   **[ClinicalTrials.gov Data API - Study Data Structure](https://clinicaltrials.gov/data-api/about-api/study-data-structure)**:
    This documentation outlines the structure of the data returned by the API. It helps you understand what information is available for each study (e.g., NCTId, BriefTitle, Conditions, etc.) and how it's organized within the JSON or CSV responses.

*   **[https://clinicaltrials.gov/api/v2/studies](https://clinicaltrials.gov/api/v2/studies) (Example API Endpoint)**:
    This is an example of a base URL for the ClinicalTrials.gov API (version 2). It's the primary endpoint for fetching study data. You would append parameters to this URL to specify your search criteria, fields, page size, and more, as demonstrated in the Python code above.

## Setup

In [ ]:
# Mount colab
if True:
    import os
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

# Env keys
# ...

# Paths
from pathlib import Path
PROJECT_ROOT: Path = Path(f"/content/drive/MyDrive/colab_notebooks/projects/trial_translator_project/v1.2")
OUTPUT_FOLDER_PATH: Path = PROJECT_ROOT / "data"

In [ ]:
# Install reqs
if True:
    %pip install -r {PROJECT_ROOT / "requirements.txt"}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.2/511.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.13.0
    Uninstalling jiter-0.13.0:
      Successfully uninstalled jiter-0.13.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-

In [ ]:
# Imports and reloads

# -- Base imports --
import os
import sys
import importlib
import pandas as pd
from pprint import pprint

# Add the PROJECT_ROOT directory to the Python path to import modules
src_path = PROJECT_ROOT
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# -- project imports --
import src.clients.clinical_trials

# Reload all relevant modules
importlib.reload(src.clients.clinical_trials)

print("✅All relevant modules have been imported/reloaded.")

✅All relevant modules have been imported/reloaded.


## Get data with API

In [ ]:
from src.clients.clinical_trials import (
    ClinicalTrialsApiClient,
    ClientConfiguration,
    SearchConfig,
    CFG
)

# Instantiate configs searching for diabetes and depression
config = CFG(
    search=SearchConfig(
        search_expr=["Oncology", "Cardiovascular", "Depression", "Infection"],
        overall_status="COMPLETED",
        )
    )
pprint(config.to_dict())

{'client': {'base_url': 'https://clinicaltrials.gov/api/v2/studies',
            'headers': {'Accept': 'application/json',
                        'User-Agent': 'ClinicalTrialsProject/1.0'},
            'min_delay': 1.2,
            'retries': 3},
 'search': {'fields': ['NCTId',
                       'BriefTitle',
                       'OverallStatus',
                       'BriefSummary',
                       'DetailedDescription',
                       'Condition'],
            'overall_status': 'COMPLETED',
            'page_size': 500,
            'search_expr': ['Oncology',
                            'Cardiovascular',
                            'Depression',
                            'Infection'],
            'total_to_collect': 2000}}


In [ ]:
# Initialize api class and get data
api = ClinicalTrialsApiClient(client_config=config.client)
config.search.total_to_collect = 1000 # downsample
studies = api.get_studies(search_config=config.search, search_mode="INDIVIDUAL")

Fetching studies.
Target: 1000 | Page size: 500 | Search Mode: INDIVIDUAL
Sleeping for 1.19 seconds.
Sleeping for 1.19 seconds.
Sleeping for 1.19 seconds.
Total items collected via pagination: 1000


In [ ]:
# Data check
pprint(studies[0])

{'protocolSection': {'identificationModule': {'nctId': 'NCT04414150',
   'briefTitle': 'A Trial of SHR-1802 in Patients With Failure of Standard Treatment for Advanced Malignant Tumours'},
  'statusModule': {'overallStatus': 'COMPLETED'},
  'descriptionModule': {'briefSummary': 'This is the first study to test SHR-1802 in humans. The primary purpose of this study is to see if SHR-1802 is safe and tolerable for patients with locally advanced/unresectable or metastatic malignancies that are refractory to available therapy or for which no standard therapy is available.'},
  'conditionsModule': {'conditions': ['Malignant Tumours']}},
 'search_term_used': 'Oncology'}

### Preprocess data

In [ ]:
# Normalize json and display
df = pd.json_normalize(studies)
display(df)

# Count null empty values
print("="*100, "\nNull values:")
display(df.isnull().sum())

,search_term_used,protocolSection.identificationModule.nctId,protocolSection.identificationModule.briefTitle,protocolSection.statusModule.overallStatus,protocolSection.descriptionModule.briefSummary,protocolSection.conditionsModule.conditions,protocolSection.descriptionModule.detailedDescription
0,Oncology,NCT04414150,A Trial of SHR-1802 in Patients With Failure o...,COMPLETED,This is the first study to test SHR-1802 in hu...,[Malignant Tumours],NaN
1,Oncology,NCT04660695,Nasobiliary Drain Assisted EUS-guided Gastroen...,COMPLETED,Gastric outlet obstruction in malignant diseas...,"[Gastric Outlet Obstruction, Gastric Cancer, P...",DESIGN\n\nProspective multicenter case series\...
2,Oncology,NCT03435120,Quality of Life Study Breakthrough Cancer Pain...,COMPLETED,"In cancer patients, pain is one of the most co...",[Breakthrough Pain],NaN
3,Oncology,NCT03135769,Avelumab in Chemo-resistant Gestational Tropho...,COMPLETED,Gestational trophoblastic neoplasias (GTN) are...,[Gestational Trophoblastic Neoplasias (GTN)],NaN
4,Oncology,NCT05237206,Study of SUPLEXA in Patients With Metastatic S...,COMPLETED,"This Phase 1, first-in-human (FIH), open-label...",[Oncology],"This is a FIH Phase 1, non-comparative, open-l..."
...,...,...,...,...,...,...,...
995,Infection,NCT03656198,Non-specific Effects of Rabies Vaccine,COMPLETED,Vaccines work by stimulating the body to produ...,"[Influenza-like Illness, Upper Respiratory Dis...",NaN
996,Infection,NCT04067869,Cytomegalovirus Infection in Patients With HIV...,COMPLETED,The purpose of this study is to assess the lev...,"[Cytomegalovirus Infections, HIV-1-infection]",T cell activation plays a central role in HIV ...
997,Infection,NCT00232518,Randomised Controlled Clinical Trials of the E...,COMPLETED,There has been considerable debate over the la...,"[Rhinitis, Allergic, Perennial, Hayfever, Alle...",Epidemiological evidence suggests that human h...
998,Infection,NCT05005130,Task-shifted Adaptation of the WHO-PEN Interve...,COMPLETED,This mixed-methods formative research study ai...,"[Non Communicable Diseases, HIV]",Using local data and implementation science th...


Null values:


,0
search_term_used,0
protocolSection.identificationModule.nctId,0
protocolSection.identificationModule.briefTitle,0
protocolSection.statusModule.overallStatus,0
protocolSection.descriptionModule.briefSummary,0
protocolSection.conditionsModule.conditions,0
protocolSection.descriptionModule.detailedDescription,263


In [ ]:
# Rename columns
new_col_names = [name.split('.')[-1] for name in df.columns]
df.columns = new_col_names

### Checks and diagnostics

In [ ]:
SANITY_CHECK =True
if SANITY_CHECK:
    # Select relevant columns for text search
    text_columns_to_search = ['briefTitle', 'briefSummary', 'conditions', 'detailedDescription']

    # Create a temporary copy to avoid modifying the original DataFrame directly yet
    temp_df_for_search = df[text_columns_to_search].copy()

    # Preprocess columns: convert lists to strings and fill NaN
    for col in text_columns_to_search:
        if col == 'conditions': # Special handling for list of strings
            temp_df_for_search[col] = temp_df_for_search[col].apply(lambda x: " ".join(x) if isinstance(x, list) else (str(x) if pd.notna(x) else ""))
        else:
            temp_df_for_search[col] = temp_df_for_search[col].astype(str).fillna("")

    # Concatenate all selected text columns into a single string for each row
    temp_df_for_search['combined_text'] = temp_df_for_search.apply(lambda row: ' '.join(row.values), axis=1)

    # Check if the combined text contains the keyword (case-insensitive)
    for keyword in config.search.search_expr:
        print(f"Hits for {keyword}: {temp_df_for_search.combined_text.str.contains(keyword[:-2], case=False).sum()}")

Hits for Oncology: 132
Hits for Cardiovascular: 173
Hits for Depression: 250
Hits for Infection: 226


In [ ]:
# Length diagnostics
df["detailed_len"] = df["detailedDescription"].fillna("").apply(lambda x: x.split()).str.len()
df["brief_len"] = df["briefSummary"].fillna("").apply(lambda x: x.split()).str.len()
df[["detailed_len","brief_len"]].describe()

,detailed_len,brief_len
count,1000.000000,1000.000000
mean,232.900000,103.178000
std,361.587674,90.215338
min,0.000000,9.000000
25%,0.000000,37.000000
50%,141.000000,71.000000
75%,312.250000,138.000000
max,4317.000000,577.000000


## Save data

In [ ]:
df.to_csv(f"{OUTPUT_FOLDER_PATH}/clinical-trials-bronze-raw.csv", index=False)

# Requirements

In [ ]:
if True:
  !pip freeze

absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.3
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.12.1
anywidget==0.9.21
apsw==3.51.3.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.3.16.0.53.33
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.1.0
Authlib==1.6.9
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.38.0
bigquery-magics==0.10.3
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.1.2
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.2.25
c

```markdown
absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.3
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.12.1
anywidget==0.9.21
apsw==3.51.3.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.3.16.0.53.33
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.1.0
Authlib==1.6.9
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.38.0
bigquery-magics==0.10.3
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.1.2
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.2.25
cffi==2.0.0
chardet==5.2.0
charset-normalizer==3.4.6
clarabel==0.11.1
click==8.3.1
click-plugins==1.1.1.2
cligj==0.7.2
cloudpathlib==0.23.0
cloudpickle==3.1.2
cmake==3.31.10
cmdstanpy==1.3.0
colorcet==3.1.0
colorlover==0.3.0
colour==0.1.5
community==1.0.0b1
confection==0.1.5
cons==0.4.7
contourpy==1.3.3
cramjam==2.11.0
cryptography==43.0.3
cufflinks==0.17.3
curl_cffi==0.14.0
cvxopt==1.3.2
cvxpy==1.6.7
cycler==0.12.1
cyipopt==1.5.0
cymem==2.0.13
Cython==3.0.12
dask==2026.1.2
dataproc-spark-connect==1.0.2
datasets==4.0.0
db-dtypes==1.5.0
dbus-python==1.2.18
debugpy==1.8.15
decorator==4.4.2
defusedxml==0.7.1
deprecation==2.1.0
diffusers==0.37.0
dill==0.3.8
diskcache==5.6.3
distilabel==1.5.3
distro==1.9.0
dlib==19.24.6
dm-tree==0.1.9
docstring_parser==0.17.0
docutils==0.21.2
dopamine_rl==4.1.2
duckdb==1.3.2
earthengine-api==1.5.24
easydict==1.13
editdistance==0.8.1
eerepr==0.1.2
einops==0.8.2
en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl#sha256=1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85
entrypoints==0.4
esda==2.8.2
et_xmlfile==2.0.0
etils==1.14.0
etuples==0.3.10
Farama-Notifications==0.0.4
fastai==2.8.7
fastapi==0.135.1
fastcore==1.12.26
fastdownload==0.0.7
fastjsonschema==2.21.2
fastlite==0.2.4
fastprogress==1.1.5
fasttransform==0.0.2
fastuuid==0.14.0
ffmpy==1.0.0
filelock==3.25.2
fiona==1.10.1
firebase-admin==6.9.0
Flask==3.1.3
flatbuffers==25.12.19
flax==0.11.2
folium==0.20.0
fonttools==4.62.1
fqdn==1.5.1
frozendict==2.4.7
frozenlist==1.8.0
fsspec==2025.3.0
future==1.0.0
gast==0.7.0
gcsfs==2025.3.0
GDAL==3.8.4
gdown==5.2.1
geemap==0.35.3
geocoder==1.38.1
geographiclib==2.1
geopandas==1.1.3
geopy==2.4.1
giddy==2.3.8
gin-config==0.5.0
gitdb==4.0.12
GitPython==3.1.46
glob2==0.7
google==3.0.0
google-adk==1.27.1
google-ai-generativelanguage==0.6.15
google-api-core==2.30.0
google-api-python-client==2.192.0
google-auth==2.47.0
google-auth-httplib2==0.3.0
google-auth-oauthlib==1.3.0
google-cloud-aiplatform==1.141.0
google-cloud-appengine-logging==1.8.0
google-cloud-audit-log==0.4.0
google-cloud-bigquery==3.40.1
google-cloud-bigquery-connection==1.20.0
google-cloud-bigquery-storage==2.36.2
google-cloud-bigtable==2.35.0
google-cloud-core==2.5.0
google-cloud-dataplex==2.16.0
google-cloud-dataproc==5.25.0
google-cloud-datastore==2.23.0
google-cloud-discoveryengine==0.13.12
google-cloud-firestore==2.25.0
google-cloud-functions==1.22.0
google-cloud-iam==2.21.0
google-cloud-language==2.19.0
google-cloud-logging==3.14.0
google-cloud-monitoring==2.29.1
google-cloud-pubsub==2.36.0
google-cloud-resource-manager==1.16.0
google-cloud-secret-manager==2.26.0
google-cloud-spanner==3.63.0
google-cloud-speech==2.37.0
google-cloud-storage==3.9.0
google-cloud-trace==1.18.0
google-cloud-translate==3.24.0
google-colab @ file:///colabtools/dist/google_colab-1.0.0.tar.gz
google-crc32c==1.8.0
google-genai==1.67.0
google-generativeai==0.8.6
google-pasta==0.2.0
google-resumable-media==2.8.0
googleapis-common-protos==1.73.0
googledrivedownloader==1.1.0
gradio==5.50.0
gradio_client==1.14.0
grain==0.2.16
graphviz==0.21
greenlet==3.3.2
groovy==0.1.2
groq==1.1.2
grpc-google-iam-v1==0.14.3
grpc-interceptor==0.15.4
grpcio==1.78.0
grpcio-status==1.71.2
grpclib==0.4.9
gspread==6.2.1
gspread-dataframe==4.0.0
gym==0.25.2
gym-notices==0.1.0
gymnasium==1.2.3
h11==0.16.0
h2==4.3.0
h5netcdf==1.8.1
h5py==3.16.0
hdbscan==0.8.41
hf-xet==1.4.2
highspy==1.13.1
holidays==0.93
holoviews==1.22.1
hpack==4.1.0
html5lib==1.1
httpcore==1.0.9
httpimport==1.4.1
httplib2==0.31.2
httptools==0.7.1
httpx==0.28.1
httpx-sse==0.4.3
huggingface_hub==1.6.0
humanize==4.15.0
hyperframe==6.1.0
hyperopt==0.2.7
ibis-framework==9.5.0
idna==3.11
ImageIO==2.37.3
imageio-ffmpeg==0.6.0
imagesize==2.0.0
imbalanced-learn==0.14.1
immutabledict==4.3.1
importlib_metadata==8.7.1
importlib_resources==6.5.2
imutils==0.5.4
inequality==1.1.2
inflect==7.5.0
iniconfig==2.3.0
instructor==1.14.5
intel-cmplr-lib-ur==2025.3.2
intel-openmp==2025.3.2
ipyevents==2.0.4
ipyfilechooser==0.6.0
ipykernel==6.17.1
ipyleaflet==0.20.0
ipyparallel==8.8.0
ipython==7.34.0
ipython-genutils==0.2.0
ipython-sql==0.5.0
ipytree==0.2.2
ipywidgets==7.7.1
isoduration==20.11.0
itsdangerous==2.2.0
jaraco.classes==3.4.0
jaraco.context==6.1.1
jaraco.functools==4.4.0
jax==0.7.2
jaxlib==0.7.2
jeepney==0.9.0
jieba==0.42.1
Jinja2==3.1.6
jiter==0.11.1
joblib==1.5.3
jsonpatch==1.33
jsonpickle==4.1.1
jsonpointer==3.0.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-console==6.6.3
jupyter-events==0.12.0
jupyter-leaflet==0.20.0
jupyter_client==7.4.9
jupyter_core==5.9.1
jupyter_kernel_gateway @ git+https://github.com/googlecolab/kernel_gateway@b134e9945df25c2dcb98ade9129399be10788671
jupyter_server==2.14.0
jupyter_server_terminals==0.5.4
jupyterlab_pygments==0.3.0
jupyterlab_widgets==3.0.16
jupytext==1.19.1
kaggle==2.0.0
kagglehub==1.0.0
kagglesdk==0.1.16
keras==3.13.2
keras-hub==0.26.0
keras-nlp==0.26.0
keyring==25.7.0
keyrings.google-artifactregistry-auth==1.1.2
kiwisolver==1.5.0
langchain==1.2.12
langchain-core==1.2.19
langgraph==1.1.2
langgraph-checkpoint==4.0.1
langgraph-prebuilt==1.0.8
langgraph-sdk==0.3.11
langsmith==0.7.18
lark==1.3.1
launchpadlib==1.10.16
lazr.restfulclient==0.14.4
lazr.uri==1.0.6
lazy-loader==0.5
libclang==18.1.1
libpysal==4.14.1
librosa==0.11.0
lightgbm==4.6.0
linkify-it-py==2.1.0
litellm==1.82.6
llvmlite==0.43.0
locket==1.0.0
logical-unification==0.4.7
lxml==6.0.2
Mako==1.3.10
mapclassify==2.10.0
Markdown==3.10.2
markdown-it-py==4.0.0
MarkupSafe==3.0.3
matplotlib==3.10.0
matplotlib-inline==0.2.1
matplotlib-venn==1.1.2
mcp==1.26.0
mdit-py-plugins==0.5.0
mdurl==0.1.2
mgwr==2.2.1
miniKanren==1.0.5
missingno==0.5.2
mistune==3.2.0
mizani==0.13.5
mkl==2025.3.1
ml_dtypes==0.5.4
mlxtend==0.23.4
mmh3==5.2.1
momepy==0.11.0
more-itertools==10.8.0
moviepy==1.0.3
mpmath==1.3.0
msgpack==1.1.2
multidict==6.7.1
multipledispatch==1.0.0
multiprocess==0.70.16
multitasking==0.0.12
murmurhash==1.0.15
music21==9.9.1
namex==0.1.0
narwhals==2.18.0
natsort==8.4.0
nbclassic==1.3.3
nbclient==0.10.4
nbconvert==7.17.0
nbformat==5.10.4
ndindex==1.10.1
nest-asyncio==1.6.0
networkx==3.6.1
nibabel==5.4.2
nltk==3.9.1
notebook==6.5.7
notebook_shim==0.2.4
numba==0.60.0
numexpr==2.14.1
numpy==2.0.2
nvidia-nccl-cu12==2.29.7
oauth2client==4.1.3
oauthlib==3.3.1
omegaconf==2.3.0
onemkl-license==2025.3.1
openai==2.28.0
opencv-contrib-python==4.13.0.92
opencv-python==4.13.0.92
opencv-python-headless==4.13.0.92
openpyxl==3.1.5
opentelemetry-api==1.38.0
opentelemetry-exporter-gcp-logging==1.11.0a0
opentelemetry-exporter-gcp-monitoring==1.11.0a0
opentelemetry-exporter-gcp-trace==1.11.0
opentelemetry-exporter-otlp-proto-common==1.38.0
opentelemetry-exporter-otlp-proto-http==1.38.0
opentelemetry-proto==1.38.0
opentelemetry-resourcedetector-gcp==1.11.0a0
opentelemetry-sdk==1.38.0
opentelemetry-semantic-conventions==0.59b0
opt_einsum==3.4.0
optax==0.2.7
optree==0.19.0
orbax-checkpoint==0.11.33
orjson==3.11.7
ormsgpack==1.12.2
osqp==1.1.1
overrides==7.7.0
packaging==26.0
pandas==2.2.2
pandas-datareader==0.10.0
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
pandocfilters==1.5.1
panel==1.8.10
param==2.3.2
parso==0.8.6
parsy==2.2
partd==1.4.2
pathlib_abc==0.5.2
patsy==1.0.2
peewee==4.0.2
peft==0.18.1
pexpect==4.9.0
pickleshare==0.7.5
pillow==11.3.0
platformdirs==4.9.4
plotly==5.24.1
plotnine==0.14.5
pluggy==1.6.0
plum-dispatch==2.7.1
pointpats==2.5.5
polars==1.35.2
polars-runtime-32==1.35.2
pooch==1.9.0
portalocker==3.2.0
portpicker==1.5.2
preshed==3.0.12
prettytable==3.17.0
proglog==0.1.12
progressbar2==4.5.0
prometheus_client==0.24.1
promise==2.3
prompt_toolkit==3.0.52
propcache==0.4.1
prophet==1.3.0
proto-plus==1.27.1
protobuf==5.29.6
psutil==5.9.5
psycopg2==2.9.11
psygnal==0.15.1
ptyprocess==0.7.0
PuLP==3.3.0
py-cpuinfo==9.0.0
py4j==0.10.9.9
pyarrow==18.1.0
pyasn1==0.6.3
pyasn1_modules==0.4.2
pycairo==1.29.0
pycocotools==2.0.11
pycparser==3.0
pycryptodomex==3.23.0
pydantic==2.12.3
pydantic-settings==2.13.1
pydantic_core==2.41.4
pydata-google-auth==1.9.1
pydot==4.0.1
pydotplus==2.0.2
PyDrive2==1.21.3
pydub==0.25.1
pyerfa==2.0.1.5
pygame==2.6.1
pygit2==1.19.1
Pygments==2.19.2
PyGObject==3.48.2
pyiceberg==0.11.1
PyJWT==2.12.1
pymc==5.28.1
pynndescent==0.6.0
pyogrio==0.12.1
pyomo==6.10.0
PyOpenGL==3.1.10
pyOpenSSL==24.2.1
pyparsing==3.3.2
pyperclip==1.11.0
pyproj==3.7.2
pyroaring==1.0.3
pysal==25.7
pyshp==3.0.3
PySocks==1.7.1
pyspark==4.0.2
pytensor==2.38.2
pytest==8.4.2
python-apt==0.0.0
python-box==7.4.1
python-dateutil==2.9.0.post0
python-dotenv==1.2.2
python-fasthtml==0.12.50
python-json-logger==4.0.0
python-louvain==0.16
python-multipart==0.0.22
python-slugify==8.0.4
python-snappy==0.7.3
python-utils==3.9.1
pytz==2025.2
pyviz_comms==3.0.6
PyWavelets==1.9.0
PyYAML==6.0.3
pyzmq==26.2.1
quantecon==0.11.1
rasterio==1.5.0
rasterstats==0.20.0
ratelim==0.1.6
referencing==0.37.0
regex==2025.11.3
requests==2.32.4
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0
requirements-parser==0.9.0
rfc3339-validator==0.1.4
rfc3986-validator==0.1.1
rfc3987-syntax==1.1.0
rich==13.9.4
roman-numerals==4.1.0
roman-numerals-py==4.1.0
rpds-py==0.30.0
rpy2==3.5.17
rsa==4.9.1
rtree==1.4.1
ruff==0.15.6
safehttpx==0.1.7
safetensors==0.7.0
scikit-image==0.25.2
scikit-learn==1.6.1
scipy==1.16.3
scooby==0.11.0
scs==3.2.11
seaborn==0.13.2
SecretStorage==3.5.0
segregation==2.5.3
semantic-version==2.10.0
Send2Trash==2.1.0
sentence-transformers==5.3.0
sentencepiece==0.2.1
sentry-sdk==2.54.0
setuptools==75.2.0
shap==0.51.0
shapely==2.1.2
shellingham==1.5.4
simple-parsing==0.1.8
simplejson==3.20.2
simsimd==6.5.16
six==1.17.0
sklearn-compat==0.1.5
sklearn-pandas==2.2.0
slicer==0.0.8
smart_open==7.5.1
smmap==5.0.3
sniffio==1.3.1
snowballstemmer==3.0.1
soundfile==0.13.1
soupsieve==2.8.3
soxr==1.0.0
spacy==3.8.11
spacy-legacy==3.0.12
spacy-loggers==1.0.5
spaghetti==1.7.6
spanner-graph-notebook==1.1.8
spglm==1.1.0
Sphinx==8.2.3
sphinxcontrib-applehelp==2.0.0
sphinxcontrib-devhelp==2.0.0
sphinxcontrib-htmlhelp==2.1.0
sphinxcontrib-jsmath==1.0.1
sphinxcontrib-qthelp==2.0.0
sphinxcontrib-serializinghtml==2.0.0
spint==1.0.7
splot==1.1.7
spopt==0.7.0
spreg==1.8.5
SQLAlchemy==2.0.48
sqlalchemy-spanner==1.17.2
sqlglot==25.20.2
sqlparse==0.5.5
srsly==2.5.2
sse-starlette==3.3.2
stanio==0.5.1
starlette==0.52.1
statsmodels==0.14.6
strictyaml==1.7.3
stringzilla==4.6.0
stumpy==1.13.0
sympy==1.14.0
tables==3.10.2
tabulate==0.9.0
tbb==2022.3.1
tblib==3.2.2
tcmlib==1.4.1
tenacity==9.1.4
tensorboard==2.19.0
tensorboard-data-server==0.7.2
tensorflow==2.19.0
tensorflow-datasets==4.9.9
tensorflow-hub==0.16.1
tensorflow-metadata==1.17.3
tensorflow-probability==0.25.0
tensorflow-text==2.19.0
tensorflow_decision_forests==1.12.0
tensorstore==0.1.82
termcolor==3.3.0
terminado==0.18.1
text-unidecode==1.3
textblob==0.19.0
tf-slim==1.1.0
tf_keras==2.19.0
thinc==8.3.10
threadpoolctl==3.6.0
tifffile==2026.3.3
tiktoken==0.12.0
timm==1.0.25
tinycss2==1.4.0
tobler==0.13.0
tokenizers==0.22.2
toml==0.10.2
tomlkit==0.13.3
toolz==0.12.1
torch @ https://download.pytorch.org/whl/cpu/torch-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
torchao==0.10.0
torchaudio @ https://download.pytorch.org/whl/cpu/torchaudio-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
torchcodec @ https://download.pytorch.org/whl/cpu/torchcodec-0.10.0-cp312-cp312-manylinux_2_28_x86_64.whl
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision @ https://download.pytorch.org/whl/cpu/torchvision-0.25.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
tornado==6.5.1
tqdm==4.67.3
traitlets==5.7.1
traittypes==0.2.3
transformers==5.0.0
treescope==0.1.10
tsfresh==0.21.1
tweepy==4.16.0
typeguard==4.5.1
typer==0.24.1
typer-slim==0.24.0
types-pytz==2026.1.1.20260304
types-setuptools==82.0.0.20260210
typing-inspection==0.4.2
typing_extensions==4.15.0
tzdata==2025.3
tzlocal==5.3.1
uc-micro-py==2.0.0
umap-learn==0.5.11
umf==1.0.3
universal_pathlib==0.3.10
uri-template==1.3.0
uritemplate==4.2.0
urllib3==2.5.0
uuid_utils==0.14.1
uvicorn==0.42.0
uvloop==0.22.1
vega-datasets==0.9.0
wadllib==1.3.6
wandb==0.25.1
wasabi==1.1.3
watchdog==6.0.0
watchfiles==1.1.1
wcwidth==0.6.0
weasel==0.4.3
webcolors==25.10.0
webencodings==0.5.1
websocket-client==1.9.0
websockets==15.0.1
Werkzeug==3.1.6
wheel==0.46.3
widgetsnbextension==3.6.10
wordcloud==1.9.6
wrapt==2.1.2
wurlitzer==3.1.1
xarray==2025.12.0
xarray-einstats==0.10.0
xgboost==3.2.0
xlrd==2.0.2
xxhash==3.6.0
xyzservices==2025.11.0
yarl==1.23.0
ydf==0.15.0
yellowbrick==1.5
yfinance==0.2.66
zipp==3.23.0
zstandard==0.25.0

```